In [1]:
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

In [2]:
df = pd.read_csv("izban_data.csv")

In [3]:
df = df.drop("date_hours", axis=1)

In [5]:
le_station = LabelEncoder()
df["station"] = le_station.fit_transform(df["station"])

le_target = LabelEncoder()
df["density_class"] = le_target.fit_transform(df["density_class"])

In [6]:
X = df.drop("density_class", axis=1)
y = df["density_class"]

In [7]:
scaler = StandardScaler()

In [8]:
X_scaled = scaler.fit_transform(X)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [10]:
gb_model = GradientBoostingClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [2, 3, 4, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.8, 0.9, 1.0]
}

In [11]:
random_search = RandomizedSearchCV(estimator=gb_model, param_distributions=param_grid, n_iter=50, cv=5, scoring='accuracy', n_jobs=-1, random_state=42)

In [12]:
random_search.fit(X_train, y_train)
print("Best Params:")
print(random_search.best_params_)

print("\nBest CV Score:")
print(random_search.best_score_)

Best Params:
{'subsample': 1.0, 'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 5, 'learning_rate': 0.01}

Best CV Score:
0.9113870162030178


In [ ]:
export_data = {
    'model': random_search.best_estimator_,
    'scaler': scaler,
    'le_station': le_station,
    'le_target': le_target
}

with open('izban_model.pkl', 'wb') as file:
    pickle.dump(export_data, file)

print("The model and all preprocessors have been successfully saved as 'izban_model.pkl'!")